# Risk reconstruction: a short walkthrough

Numerical companion to *Reconstructing risk preferences from data* by Anastasis Kratsios, Silvana M. Pesenti, and Philipp Zimmermann.

Install this repository first with `python -m pip install -e ".[notebook]"`, select that environment as the notebook kernel, and run all cells. All code is imported from the package, so there is one source of truth. No API key, dataset, or GPU is needed.

The default notebook runs a small finite experiment. The full paper commands appear below.

In [1]:
import os

for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(key, "1")
from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import pandas as pd
from risk_measure_reconstruction import (
    FiniteModel,
    FiniteGeometry,
    PopulationCallReconstructor,
    population_call_risk,
    common_lipschitz_budget,
)
from risk_measure_reconstruction.provenance import runtime_versions

runtime_versions()["packages"]

{'numpy': '2.3.5', 'scipy': '1.17.0', 'pandas': '2.2.3', 'clarabel': '0.11.1', 'mpmath': '1.3.0', 'matplotlib': '3.10.8'}

## Fit one latent ES preference

The payoff is $X_K=-(S_1-K)_+$, and its positive loss is $-X_K$. Here strikes cross the stock support, creating a nontrivial finite example. This is a tutorial example, separate from the paper's finite design.

In [2]:
stock = FiniteModel(np.linspace(0.0, 2.0, 25))
training_strikes = np.linspace(0.1, 1.9, 20)
labels = population_call_risk(stock, training_strikes, "es", 0.75).value
fit = PopulationCallReconstructor(
    FiniteGeometry(stock, training_strikes), labels, common_lipschitz_budget()
)
training_certificate = fit.certify_training()
query = np.array([0.05, 0.55, 1.05, 1.95])
pred = fit.predict(query)
truth = population_call_risk(stock, query, "es", 0.75).value
pd.DataFrame(
    {
        "strike": query,
        "latent ES": truth,
        "reconstruction": pred.value,
        "lower": pred.lower,
        "upper": pred.upper,
        "objective gap": pred.gap,
    }
)

,strike,latent ES,reconstruction,lower,upper,objective gap
0,0.05,1.730,1.801994,1.801994,1.801994,4.440892e-16
1,0.55,1.230,1.237215,1.237215,1.237215,1.423306e-13
2,1.05,0.730,0.732260,0.732260,0.732260,2.773337e-13
3,1.95,0.008,0.090661,0.090661,0.090661,4.163336e-17


## Run the paper's finite design at smoke-test size

The paper uses 1,000 equally probable stock states in $[1,10]$ and fixed strike bands within $[0,1]$. Every payoff is then $X_K=K-S_1$, so exact interpolation plus cash additivity gives zero error in real arithmetic. Tiny reported MSEs are floating-point residuals.

This cell runs two parameter draws per family with 20 training strikes. The saved settings and source hash protect checkpoints against incompatible reuse; choose a new output directory after changing settings or code.

In [3]:
roots = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in roots if (p / "pyproject.toml").exists()), Path.cwd())
out = root / "results" / "notebook-smoke"
run = subprocess.run(
    [
        sys.executable,
        "-m",
        "risk_measure_reconstruction",
        "smoke",
        "--models",
        "Finite",
        "--out",
        str(out),
    ],
    capture_output=True,
    text=True,
)
if run.returncode:
    raise RuntimeError(run.stdout + run.stderr)
print("Finite smoke experiment completed and validated.")
pd.read_csv(out / "table_finite.csv")

Finite smoke experiment completed and validated.


,Risk measure,Train MSE,Test MSE,Extrapolation TEST ratio
0,Expected Shortfall,0.000e+00 ± 0.000e+00,4.733e-30 ± 2.231e-30,1.588e-30 ± 2.246e-30
1,Power spectral,0.000e+00 ± 0.000e+00,1.578e-30 ± 0.000e+00,3.130e-30 ± 2.440e-30
2,Exponential spectral,0.000e+00 ± 0.000e+00,7.889e-30 ± 1.116e-29,2.707e-30 ± 1.763e-30
3,Wang,0.000e+00 ± 0.000e+00,3.550e-30 ± 2.231e-30,2.304e-31 ± 6.511e-32


In [4]:
json.loads((out / "validation.json").read_text())

{'passed': True, 'draw_count': 8, 'prediction_count': 192, 'max_objective_gap': 5.329070518200751e-15, 'min_objective_gap': -1.7763568394002505e-15, 'minimum_benchmark_mse': 0.6499023125691319, 'nonfinite_ratio_count': 0, 'max_risk_error_indicator': 0.0, 'finite_cash_additivity_residual': 5.329070518200751e-15, 'checks': ['draws and strikes', 'per-draw MSEs and ratios', 'sample SDs (ddof=1)', 'primal/dual objective gaps', 'finite cash-additivity']}

## Full reproduction

From the repository root, run:

```bash
risk-reconstruct reproduce --config configs/paper.json --out results/paper
risk-reconstruct ablation --config configs/ablation.json --out results/ablation
risk-reconstruct validate --input results/paper
```

The main command evaluates GBM, Heston, and the finite law with 30 draws per risk family, then writes raw predictions, metrics, LaTeX tables, and numerical diagnostics. The ablation writes two PDF figures and their source data.

See `docs/method.md` for the mathematical contract and `docs/paper_alignment.md` for draft clarifications. In particular, the conic solver generates the tables; the randomized dual is an optional approximation, and the general reconstruction does not automatically enforce monotonicity.